In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

dbutils.widgets.text("pipeline_name", "")
dbutils.widgets.text("status", "")
dbutils.widgets.text("run_id", "")

pipeline_name = dbutils.widgets.get("pipeline_name")
status = dbutils.widgets.get("status")
run_id = dbutils.widgets.get("run_id")

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("run_id", StringType(), True)
])

data = [(pipeline_name, status, run_id)]

df_run_log = spark.createDataFrame(data, schema) \
    .withColumn("log_timestamp", current_timestamp())

storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(
    scope="aml-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)
print("pipeline_name =", pipeline_name)
print("status =", status)
print("run_id =", run_id)


df_run_log.write.mode("append").parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_run_logs"
)

df_run_log.show(truncate=False)

In [0]:
print("pipeline_name =", pipeline_name)
print("status =", status)
print("run_id =", run_id)

In [0]:
storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(
    scope="aml-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)

db_run_log1 = spark.read.parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_run_logs"
)

db_run_log1.show(20)

db_run_log1.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_run_logs_csv")


df_error_log = spark.read.parquet(
    "wasbs://logs@stbankamldev.blob.core.windows.net/error_logs"
)

df_error_log.show(20)

df_error_log.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv("wasbs://logs@stbankamldev.blob.core.windows.net/error_logs_csv")

In [0]:
dbutils.fs.ls(
"wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_run_logs_csv"
)

In [0]:
storage_account = "stbankamldev"

storage_key = dbutils.secrets.get(
    scope="aml-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    storage_key
)



#pipeline run log fixed file
files = dbutils.fs.ls("wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_run_logs_csv")
part_file = [f.path for f in files if f.name.startswith("part-")][0]


dbutils.fs.cp(
    part_file, 
    "wasbs://logs@stbankamldev.blob.core.windows.net/pipeline_run_logs_fixed/pipeline_run_logs.csv"
)



